# Student Placement Prediction Using CART

**Machine Learning | Classification | Python | Jupyter Notebook**

## Objective
Build a CART classification model to predict whether a student is likely to be placed using academic performance, attendance, aptitude, coding, communication, project, internship, and backlog information.

**CART** stands for *Classification and Regression Trees*. For this classification task, the decision tree uses binary splits and the Gini impurity criterion.

> The dataset in this project is synthetically generated for academic practice. It does not represent real students or actual placement records.

## 1. Import the required libraries

Run this cell first. If a library is missing, install it using `pip install pandas numpy matplotlib scikit-learn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the dataset

Keep `student_placement_cart_dataset.csv` in the same folder as this notebook.

In [ ]:
DATA_FILE = Path("student_placement_cart_dataset.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Dataset not found. Keep student_placement_cart_dataset.csv "
        "in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values in each column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\\nStatistical summary:")
display(df.describe(include="all").T)

## 4. Check and clean the data

In [ ]:
# Remove duplicate records, if any.
df = df.drop_duplicates().copy()

# Remove the identifier because it does not help predict placement.
df = df.drop(columns=["Student_ID"])

# Check that the target column is present and has valid values.
target_column = "Placement_Status"
if target_column not in df.columns:
    raise ValueError(f"Required target column '{target_column}' is missing.")

df = df.dropna(subset=[target_column]).copy()
print("Shape after basic cleaning:", df.shape)
print("Target categories:", sorted(df[target_column].unique()))

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Count of placed and not-placed students.
placement_counts = df[target_column].value_counts()

plt.figure(figsize=(6, 4))
placement_counts.plot(kind="bar")
plt.title("Placement Status Distribution")
plt.xlabel("Placement Status")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Distribution of selected numerical features.
numeric_columns = [
    "CGPA", "Attendance_Percentage", "Aptitude_Score",
    "Coding_Score", "Communication_Score", "Projects_Completed", "Backlogs"
]
df[numeric_columns].hist(figsize=(12, 9), bins=15, edgecolor="black")
plt.suptitle("Distribution of Student Features")
plt.tight_layout()
plt.show()

## 6. Separate features (X) and target (y)

The target is `Placement_Status`. The other columns are input features.
Categorical input data (`Internship_Experience`) is converted into numeric form using one-hot encoding.

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column].map({"Not Placed": 0, "Placed": 1})

if y.isna().any():
    raise ValueError("Unexpected target labels found. Check Placement_Status values.")

X = pd.get_dummies(X, columns=["Internship_Experience"], drop_first=True, dtype=int)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
display(X.head())
print("Target mapping: 0 = Not Placed, 1 = Placed")

## 7. Split the data into training and testing sets

The model is trained on 80% of the data and evaluated on the remaining 20%. Stratification preserves the class proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(3))

## 8. Train the CART model

`criterion="gini"` selects Gini impurity to evaluate the quality of each split. `max_depth` limits tree depth to reduce overfitting. `random_state` makes the result reproducible.

In [ ]:
cart_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE
)

cart_model.fit(X_train, y_train)
print("CART model trained successfully.")

## 9. Make predictions

In [ ]:
y_pred = cart_model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.map({0: "Not Placed", 1: "Placed"}).values,
    "Predicted": pd.Series(y_pred, index=y_test.index).map(
        {0: "Not Placed", 1: "Placed"}
    ).values
})
display(results.head(10))

## 10. Evaluate the model

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Of the students predicted as placed, how many were actually placed.
- **Recall:** Of the students who were actually placed, how many the model identified.
- **F1-score:** Harmonic mean of precision and recall.

The scores are calculated on the held-out test set.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics.round(4))

print("Classification Report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Not Placed", "Placed"],
    zero_division=0
))

## 11. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Placed", "Placed"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("CART Confusion Matrix")
plt.tight_layout()
plt.show()

print("Confusion matrix layout:")
print(cm)

## 12. Visualize the CART decision tree

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    cart_model,
    feature_names=X.columns,
    class_names=["Not Placed", "Placed"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("CART Decision Tree for Student Placement Prediction")
plt.tight_layout()
plt.show()

## 13. Feature importance

In [ ]:
importance = pd.Series(
    cart_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importance.to_frame("Importance").round(4))

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.title("CART Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 14. Predict placement for a new student

Edit the example values if you want to test another student. The feature names and order are aligned with the training data automatically.

In [ ]:
new_student = pd.DataFrame([{
    "CGPA": 8.2,
    "Attendance_Percentage": 90.0,
    "Aptitude_Score": 78,
    "Coding_Score": 82,
    "Communication_Score": 75,
    "Projects_Completed": 3,
    "Backlogs": 0,
    "Internship_Experience": "Yes"
}])

new_student = pd.get_dummies(
    new_student,
    columns=["Internship_Experience"],
    drop_first=True,
    dtype=int
)
new_student = new_student.reindex(columns=X.columns, fill_value=0)

prediction = cart_model.predict(new_student)[0]
prediction_label = "Placed" if prediction == 1 else "Not Placed"

print("Predicted Placement Status:", prediction_label)

## 15. Final conclusion

A CART Decision Tree classifier was developed to predict student placement status. The dataset was inspected and prepared, categorical data was encoded, and the data was divided into training and testing sets. The model was trained using the Gini impurity criterion and evaluated with accuracy, precision, recall, F1-score, and a confusion matrix.

The decision tree visualization shows the sequence of binary decisions made by the model. Feature importance indicates which input variables the fitted tree used most.

**Note:** This is an educational demonstration using synthetic data. Its scores should not be interpreted as real-world placement performance.

## 16. Save the test predictions (optional)

In [ ]:
# Save predictions to a CSV file in the notebook's current folder.
results.to_csv("cart_test_predictions.csv", index=False)
print("Predictions saved to cart_test_predictions.csv")